# 🧠 What is Pydantic?

Pydantic is a Python library used for:
- Data validation: Making sure data is correct (e.g., an age is an integer, not a string).
- Data parsing: Automatically converting data into the right types (e.g., "123" → 123).
- Defining data models: Similar to Django models, but for any use case (especially APIs).

It is widely used in FastAPI (a modern web framework) but can be used anywhere in Python.


# Why or When Do You Need Pydantic?

You need Pydantic when:

- You're getting user input, JSON, or any external data (from an API, database, form, etc.).
- You want to make sure this data is valid and clean before using it in your application.
- You want a simple way to define structured data (like objects, but safer and cleaner).



In [2]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    email: str

# Input from external source (e.g., API or frontend)
data = {'id': '1', 'name': 'Alice', 'email': 'alice@example.com'}

user = User(**data)  # Pydantic automatically parses and validates

In [3]:
user

User(id=1, name='Alice', email='alice@example.com')

In [5]:
user.id      # 1 (converted to int from string)

# Even though id was a string, it’s converted to an integer automatically. That's parsing + validation!

1

In [6]:
user.name    # 'Alice'

'Alice'

In [7]:
# Missing 'email'
data = {'id': '1', 'name': 'Alice'}

user = User(**data) # ERROR


ValidationError: 1 validation error for User
email
  Field required [type=missing, input_value={'id': '1', 'name': 'Alice'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

## Why **data?

Suppose you have a dictionary like this:

In [9]:
# Suppose you have a dictionary like this:
data = {'id': 1, 'name': 'Alice', 'email': 'alice@example.com'}

# And a function or class like this:
class User(BaseModel):
    id: int
    name: str
    email: str

# you can create an object like this:
user = User(id=1, name='Alice', email='alice@example.com')
user

User(id=1, name='Alice', email='alice@example.com')

In [11]:
# But if you already have that same data in a dictionary, instead of writing all fields again, you can do:
user = User(**data)
user

User(id=1, name='Alice', email='alice@example.com')

This is exactly the same as writing:

In [12]:
user = User(id=data['id'], name=data['name'], email=data['email'])
user

User(id=1, name='Alice', email='alice@example.com')

Because external data (from JSON, forms, APIs) usually comes as dictionaries. Pydantic models accept keyword arguments (`key=value`), so `**data` unpacks the dictionary and passes each item as a keyword argument.

# Default value

In [13]:
class User(BaseModel):
    id: int
    name: str
    is_active: bool = True  # Default value

user = User(id=1, name='Alice')

user.is_active  # True

True

# Validators
Want to check specific rules? Use `@validator`.

In [24]:
from pydantic import BaseModel, field_validator

class User(BaseModel):
    name: str

    @field_validator('name')
    @classmethod
    def name_must_not_be_empty(cls, value):
        if not value.strip():
            raise ValueError('Name cannot be empty')
        return value

user = User(name="pydantic")
user

User(name='pydantic')

In [25]:
user = User()
user

ValidationError: 1 validation error for User
name
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

In [26]:
user = User(name="")
user

ValidationError: 1 validation error for User
name
  Value error, Name cannot be empty [type=value_error, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

In [28]:
user = User(name=" ")
user

ValidationError: 1 validation error for User
name
  Value error, Name cannot be empty [type=value_error, input_value=' ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

# Nested Models

In [15]:
class Address(BaseModel):
    city: str
    country: str

class User(BaseModel):
    name: str
    address: Address

data = {
    'name': 'Alice',
    'address': {
        'city': 'Delhi',
        'country': 'India'
    }
}

user = User(**data)

user.address.city  # Delhi

'Delhi'

# Type Conversion and Validation

In [16]:
class Product(BaseModel):
    name: str
    price: float
    in_stock: bool

data = {
    'name': 'Laptop',
    'price': '59999.99',
    'in_stock': 'true'
}

product = Product(**data)

In [17]:
product.price      # 59999.99 (converted to float)

59999.99

In [18]:
product.in_stock   # True (converted from string)

True

# optional fields or missing keys

## All Required Fields

In [30]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    email: str

data = {'id': 1, 'name': 'Alice', 'email': 'alice@example.com'}

In [31]:
user = User(**data)
user

User(id=1, name='Alice', email='alice@example.com')

In [32]:
data = {'id': 1, 'name': 'Alice'}  # Missing 'email'

user = User(**data)  # Raises ValidationError

ValidationError: 1 validation error for User
email
  Field required [type=missing, input_value={'id': 1, 'name': 'Alice'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

## Using Optional Fields

In [33]:
from typing import Optional

class User(BaseModel):
    id: int
    name: str
    email: Optional[str] = None  # Optional

data = {'id': 1, 'name': 'Alice'}

user = User(**data)
user

User(id=1, name='Alice', email=None)

In [34]:
data = {'id': 'abc', 'name': 'Alice', 'email': 'alice@example.com'}

user = User(**data)  # 'abc' is not a valid int


ValidationError: 1 validation error for User
id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='abc', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing

In [35]:
data = {'id': 1, 'name': 'Alice', 'email': 'alice@example.com', 'age': 30}

user = User(**data)
user

User(id=1, name='Alice', email='alice@example.com')

In [37]:
# Extra Fields (by default, allowed but ignored)

data = {'id': 1, 'name': 'Alice', 'email': 'alice@example.com', 'age': 30}

user = User(**data)
user

# age is silently ignored

User(id=1, name='Alice', email='alice@example.com')

**To forbid extra fields, do this:**

In [38]:
class User(BaseModel):
    id: int
    name: str
    email: str

    class Config:
        extra = 'forbid'


Now, using the same `data` with extra `age` will raise an error:

In [39]:
data = {'id': 1, 'name': 'Alice', 'email': 'alice@example.com', 'age': 30}
user = User(**data)
user

ValidationError: 1 validation error for User
age
  Extra inputs are not permitted [type=extra_forbidden, input_value=30, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/extra_forbidden